In [1]:
import pandas as pd
import plotly.express as px
import yfinance as yf
from IPython.display import display

# --- 1. Veriyi Oluşturma ve Ön İşleme ---
data = {
    'Sembol': ['NVDA', 'BRK.B', 'AAPL', 'AAPL', 'STX', 'STX', 'META', 'VOO'],
    'Tarih': ['15.10.2025', '15.10.2025', '16.10.2025', '20.10.2025', '22.10.2025', '23.10.2025', '24.10.2025', '24.10.2025'],
    'Alınan Miktar ($)': [180.00, 125.00, 150.00, 150.00, 213.13, 225.03, 311.29, 366.48], 
    'İşlem Ücreti': [1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5],
    'O anki fiyat': [179.57, 492.89, 247.39, 258.62, 213.13, 225.03, 622.57, 732.95],
    
    # İşlem tipleri İngilizce (BUY/SELL) olarak düzeltildi
    'İşlem': ['BUY', 'BUY', 'BUY', 'SELL', 'BUY', 'SELL', 'BUY', 'BUY'],
    'İşlem Dönüşü ($)': [-181.5, -126.5, -151.5, 148.5, -214.63, 223.53, -312.79, -367.98]
}

df = pd.DataFrame(data)
df['Tarih'] = pd.to_datetime(df['Tarih'], format='%d.%m.%Y')

# HİSSE ADEDİ HESAPLAMASI (Canlı P&L için zorunlu)
# Adet = Tutar / Fiyat (Basit Pozisyon Büyüklüğü)
df['Adet'] = (df['Alınan Miktar ($)'] / df['O anki fiyat']).round(3)

# P&L için sadece İşlem Dönüşü sütununu kullanıyoruz
df['Nakış Akışı'] = df['İşlem Dönüşü ($)']

# --- 2. Portföy Durumunu Belirleme (Açık Pozisyonlar) ---

# Net hisse adedini hesapla (Çözüm A için zorunlu)
net_pozisyonlar = df.groupby(['Sembol', 'İşlem']).agg({'Adet': 'sum'}).unstack(fill_value=0)
net_pozisyonlar.columns = net_pozisyonlar.columns.droplevel(0)
net_pozisyonlar['Net Adet'] = net_pozisyonlar['BUY'] - net_pozisyonlar['SELL']
net_pozisyonlar = net_pozisyonlar[net_pozisyonlar['Net Adet'] != 0].reset_index()

open_positions = net_pozisyonlar[net_pozisyonlar['Net Adet'] > 0]
open_symbols = open_positions['Sembol'].tolist()

# --- 3. YFINANCE ile Canlı Fiyat Çekme ---
print(f"Canlı fiyatı çekilecek açık pozisyonlar: {open_symbols}")

try:
    ticker_data = yf.download(open_symbols, period='1d', interval='1m', progress=False)

    current_prices = {}
    if len(open_symbols) == 1:
        current_prices[open_symbols[0]] = ticker_data['Close'].iloc[-1]
    else:
        current_prices = ticker_data['Close'].iloc[-1].to_dict()

except Exception as e:
    print(f"yfinance ile fiyat çekme hatası: {e}. Canlı fiyatlar 0.0 olarak ayarlandı.")
    current_prices = {sym: 0.0 for sym in open_symbols}


# --- 4. Gerçekleşmemiş P&L Hesaplama ---

open_df = open_positions.copy()
open_df['Canlı_Fiyat'] = open_df['Sembol'].map(current_prices)

# Ortalama Alış Fiyatını Hesapla (Sembolün ilk 'BUY' fiyatını baz alıyoruz)
average_buy_prices = df[df['İşlem'] == 'BUY'].drop_duplicates(subset='Sembol', keep='first').set_index('Sembol')['O anki fiyat']
open_df['Ort_Alış_Fiyat'] = open_df['Sembol'].map(average_buy_prices)

open_df['Gerçekleşmemiş_P&L'] = (open_df['Canlı_Fiyat'] - open_df['Ort_Alış_Fiyat']) * open_df['Net Adet']
open_df['Piyasa_Değeri'] = open_df['Canlı_Fiyat'] * open_df['Net Adet']

# Temel Metrikler
kapanmis_semboller = df[~df['Sembol'].isin(open_symbols)]['Sembol'].unique()
toplam_gerçekleşmiş_pl = df[df['Sembol'].isin(kapanmis_semboller)]['Nakış Akışı'].sum()
toplam_gerçekleşmemiş_pl = open_df['Gerçekleşmemiş_P&L'].sum()
toplam_portföy_değeri = open_df['Piyasa_Değeri'].sum()


# --- 5. Görselleştirme ve Konsol Çıktısı ---

print("\n" + "="*80)
print("             NASDAQ İŞLEM VE PORTFÖY DURUMU ANALİZİ (GÜNCELLENMİŞ)")
print("="*80)

# --- A) Portföy Özeti (Konsol) ---
print("\n--- 1. PORTFÖY ÖZETİ ---")
print(f"Toplam Gerçekleşmiş Kâr/Zarar ({', '.join(kapanmis_semboller)}): ${toplam_gerçekleşmiş_pl:.2f}")
print(f"Toplam Gerçekleşmemiş Kâr/Zarar (Canlı):  ${toplam_gerçekleşmemiş_pl:.2f}")
print(f"Açık Pozisyonların Toplam Piyasa Değeri:  ${toplam_portföy_değeri:.2f}")

print("\n--- 2. AÇIK POZİSYON DETAYI (CANLI) ---")
display(open_df[['Sembol', 'Net Adet', 'Ort_Alış_Fiyat', 'Canlı_Fiyat', 'Gerçekleşmemiş_P&L', 'Piyasa_Değeri']])

# --- B) Görselleştirme ---

# Kümülatif P&L (İşlem Dönüşü sütununa göre)
df_cum_pl = df.sort_values('Tarih').copy()
df_cum_pl['Kümülatif P&L'] = df_cum_pl['Nakış Akışı'].cumsum()

fig_cum_pl = px.line(
    df_cum_pl, 
    x='Tarih', 
    y='Kümülatif P&L', 
    title='Kümülatif Nakit Akışı/P&L Gelişimi (İşlem Dönüşü Bazında)',
    markers=True
)
fig_cum_pl.show()

# Açık Pozisyonların P&L Dağılımı (Canlı Fiyatlarla hesaplandı)
fig_unrealized = px.pie(
    open_df, 
    values='Gerçekleşmemiş_P&L', 
    names='Sembol', 
    title='Açık Pozisyonların Gerçekleşmemiş P&L Dağılımı (Canlı Fiyatlarla)'
)
fig_unrealized.show()

Canlı fiyatı çekilecek açık pozisyonlar: ['AAPL', 'BRK.B', 'META', 'NVDA', 'VOO']
YF.download() has changed argument auto_adjust default to True



1 Failed download:
['BRK.B']: YFPricesMissingError('possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")')



             NASDAQ İŞLEM VE PORTFÖY DURUMU ANALİZİ (GÜNCELLENMİŞ)

--- 1. PORTFÖY ÖZETİ ---
Toplam Gerçekleşmiş Kâr/Zarar (STX): $8.90
Toplam Gerçekleşmemiş Kâr/Zarar (Canlı):  $9.99
Açık Pozisyonların Toplam Piyasa Değeri:  $874.11

--- 2. AÇIK POZİSYON DETAYI (CANLI) ---


İşlem,Sembol,Net Adet,Ort_Alış_Fiyat,Canlı_Fiyat,Gerçekleşmemiş_P&L,Piyasa_Değeri
0,AAPL,0.026,247.39,263.570007,0.420680,6.852820
1,BRK.B,0.254,492.89,NaN,NaN,NaN
2,META,0.500,622.57,739.390015,58.410007,369.695007
3,NVDA,1.002,179.57,185.479996,5.921816,185.850956
4,VOO,0.500,732.95,623.419922,-54.765039,311.709961


In [2]:
import pandas as pd
import plotly.express as px
import yfinance as yf
from IPython.display import display

# --- 1. Veriyi Oluşturma ve Ön İşleme ---
data = {
    'Sembol': ['NVDA', 'BRK.B', 'AAPL', 'AAPL', 'STX', 'STX', 'META', 'VOO'],
    'Tarih': ['15.10.2025', '15.10.2025', '16.10.2025', '20.10.2025', '22.10.2025', '23.10.2025', '24.10.2025', '24.10.2025'],
    'Alınan Miktar ($)': [180.00, 125.00, 150.00, 150.00, 213.13, 225.03, 311.29, 366.48], 
    'İşlem Ücreti': [1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5],
    'O anki fiyat': [179.57, 492.89, 247.39, 258.62, 213.13, 225.03, 622.57, 732.95],
    'İşlem': ['BUY', 'BUY', 'BUY', 'SELL', 'BUY', 'SELL', 'BUY', 'BUY'],
    'İşlem Dönüşü ($)': [-181.5, -126.5, -151.5, 148.5, -214.63, 223.53, -312.79, -367.98]
}

df = pd.DataFrame(data)
df['Tarih'] = pd.to_datetime(df['Tarih'], format='%d.%m.%Y')

# HİSSE ADEDİ HESAPLAMASI (Canlı P&L için zorunlu)
# Adet = Tutar / Fiyat (Basit Pozisyon Büyüklüğü)
df['Adet'] = (df['Alınan Miktar ($)'] / df['O anki fiyat']).round(3)
df['Nakış Akışı'] = df['İşlem Dönüşü ($)']

# --- 2. Portföy Durumunu Belirleme (Açık ve Kapanmış Pozisyonlar) ---

# Net hisse adedini hesapla
net_pozisyonlar = df.groupby(['Sembol', 'İşlem']).agg({'Adet': 'sum'}).unstack(fill_value=0)
net_pozisyonlar.columns = net_pozisyonlar.columns.droplevel(0)
net_pozisyonlar['Net Adet'] = net_pozisyonlar['BUY'] - net_pozisyonlar['SELL']
net_pozisyonlar = net_pozisyonlar.reset_index()

# Açık (Net Adet > 0) ve Kapanmış (Net Adet == 0) sembolleri ayır
open_df_positions = net_pozisyonlar[net_pozisyonlar['Net Adet'] > 0]
closed_symbols = net_pozisyonlar[net_pozisyonlar['Net Adet'] == 0]['Sembol'].tolist()
open_symbols = open_df_positions['Sembol'].tolist()

# --- 3. YFINANCE ile Canlı Fiyat Çekme ---
print(f"Canlı fiyatı çekilecek açık pozisyonlar: {open_symbols}")

try:
    ticker_data = yf.download(open_symbols, period='1d', interval='1m', progress=False)
    current_prices = {}
    if len(open_symbols) == 1:
        current_prices[open_symbols[0]] = ticker_data['Close'].iloc[-1]
    else:
        current_prices = ticker_data['Close'].iloc[-1].to_dict()

except Exception as e:
    print(f"yfinance ile fiyat çekme hatası: {e}. Canlı fiyatlar 0.0 olarak ayarlandı.")
    current_prices = {sym: 0.0 for sym in open_symbols}


# --- 4. Gerçekleşmemiş P&L Hesaplama (Açık Pozisyonlar) ---

open_df = open_df_positions.copy()
open_df['Canlı_Fiyat'] = open_df['Sembol'].map(current_prices)

# Ortalama Alış Fiyatını Hesapla (Sembolün ilk 'BUY' fiyatını baz alıyoruz)
average_buy_prices = df[df['İşlem'] == 'BUY'].drop_duplicates(subset='Sembol', keep='first').set_index('Sembol')['O anki fiyat']
open_df['Ort_Alış_Fiyat'] = open_df['Sembol'].map(average_buy_prices)

open_df['Gerçekleşmemiş_P&L'] = (open_df['Canlı_Fiyat'] - open_df['Ort_Alış_Fiyat']) * open_df['Net Adet']
open_df['Piyasa_Değeri'] = open_df['Canlı_Fiyat'] * open_df['Net Adet']

# --- 5. Gerçekleşmiş P&L Hesaplama (Kapanmış Pozisyonlar) ---

closed_pl_df = df[df['Sembol'].isin(closed_symbols)].groupby('Sembol').agg(
    {'Nakış Akışı': 'sum'}
).reset_index().rename(columns={'Nakış Akışı': 'Gerçekleşmiş P&L ($)'})


# --- 6. Temel Metrikler ve Konsol Çıktısı ---

toplam_gerçekleşmiş_pl = closed_pl_df['Gerçekleşmiş P&L ($)'].sum()
toplam_gerçekleşmemiş_pl = open_df['Gerçekleşmemiş_P&L'].sum()
toplam_portföy_değeri = open_df['Piyasa_Değeri'].sum()


print("\n" + "="*80)
print("              NASDAQ PORTFÖY VE İŞLEM ANALİZ RAPORU")
print("="*80)

# --- A) GENEL PORTFÖY ÖZETİ ---
print("\n--- 1. GENEL P&L ÖZETİ (Tüm Zamanlar) ---")
print(f"1. Gerçekleşmiş (Kapanmış) Toplam Kâr/Zarar: ${toplam_gerçekleşmiş_pl:.2f}")
print(f"2. Gerçekleşmemiş (Açık Pozisyon) Kâr/Zarar: ${toplam_gerçekleşmemiş_pl:.2f}")
print(f"3. Açık Pozisyonların TOPLAM PİYASA DEĞERİ:  ${toplam_portföy_değeri:.2f}")

# --- B) KAPALI İŞLEMLERİN ANALİZİ ---
print("\n--- 2. KAPALI İŞLEMLER ANALİZİ (Gerçekleşmiş Kâr/Zarar) ---")
display(closed_pl_df)
# Bu tablo, hangi kapalı işlemde ne kadar kâr/zarar ettiğinizi gösterir.

# --- C) AÇIK POZİSYONLARIN ANALİZİ ---
print("\n--- 3. AÇIK POZİSYONLAR ANALİZİ (Canlı) ---")
display(open_df[['Sembol', 'Net Adet', 'Ort_Alış_Fiyat', 'Canlı_Fiyat', 'Gerçekleşmemiş_P&L', 'Piyasa_Değeri']])
# Bu tablo, canlı fiyatlara göre her bir varlığınızın durumunu gösterir.

# --- D) Görselleştirme ---

# Kümülatif P&L (İşlem Dönüşü sütununa göre)
df_cum_pl = df.sort_values('Tarih').copy()
df_cum_pl['Kümülatif P&L'] = df_cum_pl['Nakış Akışı'].cumsum()

fig_cum_pl = px.line(
    df_cum_pl, 
    x='Tarih', 
    y='Kümülatif P&L', 
    title='Kümülatif Nakit Akışı/P&L Gelişimi (İşlem Dönüşü Bazında)',
    markers=True
)
fig_cum_pl.show()

# Açık Pozisyonların P&L Dağılımı (Canlı Fiyatlarla hesaplandı)
fig_unrealized = px.pie(
    open_df, 
    values='Gerçekleşmemiş_P&L', 
    names='Sembol', 
    title='Açık Pozisyonların Gerçekleşmemiş P&L Dağılımı (Canlı Fiyatlarla)'
)
fig_unrealized.show()

Canlı fiyatı çekilecek açık pozisyonlar: ['AAPL', 'BRK.B', 'META', 'NVDA', 'VOO']



1 Failed download:
['BRK.B']: YFPricesMissingError('possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")')



              NASDAQ PORTFÖY VE İŞLEM ANALİZ RAPORU

--- 1. GENEL P&L ÖZETİ (Tüm Zamanlar) ---
1. Gerçekleşmiş (Kapanmış) Toplam Kâr/Zarar: $8.90
2. Gerçekleşmemiş (Açık Pozisyon) Kâr/Zarar: $9.72
3. Açık Pozisyonların TOPLAM PİYASA DEĞERİ:  $873.84

--- 2. KAPALI İŞLEMLER ANALİZİ (Gerçekleşmiş Kâr/Zarar) ---


,Sembol,Gerçekleşmiş P&L ($)
0,STX,8.9



--- 3. AÇIK POZİSYONLAR ANALİZİ (Canlı) ---


İşlem,Sembol,Net Adet,Ort_Alış_Fiyat,Canlı_Fiyat,Gerçekleşmemiş_P&L,Piyasa_Değeri
0,AAPL,0.026,247.39,263.470001,0.418080,6.850220
1,BRK.B,0.254,492.89,NaN,NaN,NaN
2,META,0.500,622.57,738.900024,58.165012,369.450012
3,NVDA,1.002,179.57,185.485001,5.926831,185.855971
5,VOO,0.500,732.95,623.369995,-54.790002,311.684998


In [3]:
import pandas as pd
import plotly.express as px
import yfinance as yf
from IPython.display import display

# --- 1. Veriyi Oluşturma ve Ön İşleme ---
data = {
    'Sembol': ['NVDA', 'BRK.B', 'AAPL', 'AAPL', 'STX', 'STX', 'META', 'VOO'],
    'Tarih': ['15.10.2025', '15.10.2025', '16.10.2025', '20.10.2025', '22.10.2025', '23.10.2025', '24.10.2025', '24.10.2025'],
    'Alınan Miktar ($)': [180.00, 125.00, 150.00, 150.00, 213.13, 225.03, 311.29, 366.48], 
    'İşlem Ücreti': [1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5, 1.5],
    'O anki fiyat': [179.57, 492.89, 247.39, 258.62, 213.13, 225.03, 622.57, 732.95],
    'İşlem': ['BUY', 'BUY', 'BUY', 'SELL', 'BUY', 'SELL', 'BUY', 'BUY'],
    'İşlem Dönüşü ($)': [-181.5, -126.5, -151.5, 148.5, -214.63, 223.53, -312.79, -367.98]
}

df = pd.DataFrame(data)
df['Tarih'] = pd.to_datetime(df['Tarih'], format='%d.%m.%Y')
df['Adet'] = (df['Alınan Miktar ($)'] / df['O anki fiyat']).round(3)
df['Nakış Akışı'] = df['İşlem Dönüşü ($)']

# --- 2. Portföy Durumunu Belirleme (Açık ve Kapanmış Pozisyonlar) ---

net_pozisyonlar = df.groupby(['Sembol', 'İşlem']).agg({'Adet': 'sum'}).unstack(fill_value=0)
net_pozisyonlar.columns = net_pozisyonlar.columns.droplevel(0)
net_pozisyonlar['Net Adet'] = net_pozisyonlar['BUY'] - net_pozisyonlar['SELL']
net_pozisyonlar = net_pozisyonlar.reset_index()

open_df_positions = net_pozisyonlar[net_pozisyonlar['Net Adet'] > 0]
closed_symbols = net_pozisyonlar[net_pozisyonlar['Net Adet'] == 0]['Sembol'].tolist()
open_symbols = open_df_positions['Sembol'].tolist()

# --- 3. YFINANCE ile Canlı Fiyat Çekme ve BRK.B Sorununu Çözme ---
print(f"Canlı fiyatı çekilecek açık pozisyonlar: {open_symbols}")

current_prices = {}
for symbol in open_symbols:
    try:
        # Her sembolü tek tek çekmeyi deniyoruz, bu BRK.B gibi zorlu sembollerde daha güvenilir.
        ticker = yf.Ticker(symbol)
        price_data = ticker.history(period="1d")['Close']
        if not price_data.empty:
            current_prices[symbol] = price_data.iloc[-1]
        else:
            print(f"UYARI: {symbol} için güncel fiyat bulunamadı. Fiyat 0.0 olarak ayarlandı.")
            current_prices[symbol] = 0.0
    except Exception as e:
        print(f"HATA: {symbol} için fiyat çekilemedi: {e}. Fiyat 0.0 olarak ayarlandı.")
        current_prices[symbol] = 0.0

# --- 4. Gerçekleşmemiş P&L Hesaplama (Açık Pozisyonlar) ---

open_df = open_df_positions.copy()
open_df['Canlı_Fiyat'] = open_df['Sembol'].map(current_prices)

# Ortalama Alış Fiyatını Hesapla (Sembolün ilk 'BUY' fiyatını baz alıyoruz)
average_buy_prices = df[df['İşlem'] == 'BUY'].drop_duplicates(subset='Sembol', keep='first').set_index('Sembol')['O anki fiyat']
open_df['Ort_Alış_Fiyat'] = open_df['Sembol'].map(average_buy_prices)

open_df['Gerçekleşmemiş_P&L'] = (open_df['Canlı_Fiyat'] - open_df['Ort_Alış_Fiyat']) * open_df['Net Adet']
open_df['Piyasa_Değeri'] = open_df['Canlı_Fiyat'] * open_df['Net Adet']

# --- 5. Gerçekleşmiş P&L Hesaplama (Kapanmış Pozisyonlar) ---

# Kapanmış sembollerin tüm nakit akışı toplanır. (AAPL ve STX'in net P&L'i)
# AAPL ve STX'in net adedi 0 olduğu için bu, gerçekleşmiş P&L'dir.
closed_pl_df = df[df['Sembol'].isin(closed_symbols)].groupby('Sembol').agg(
    {'Nakış Akışı': 'sum'}
).reset_index().rename(columns={'Nakış Akışı': 'Gerçekleşmiş P&L ($)'})


# --- 6. Temel Metrikler ve Görselleştirme ---

toplam_gerçekleşmiş_pl = closed_pl_df['Gerçekleşmiş P&L ($)'].sum()
toplam_gerçekleşmemiş_pl = open_df['Gerçekleşmemiş_P&L'].sum()

print("\n" + "="*80)
print("              NASDAQ PORTFÖY VE İŞLEM ANALİZ RAPORU")
print("="*80)

# --- A) GENEL PORTFÖY ÖZETİ ---
print("\n--- 1. GENEL P&L ÖZETİ (Tüm Zamanlar) ---")
print(f"1. Gerçekleşmiş (Kapanmış) Toplam Kâr/Zarar: ${toplam_gerçekleşmiş_pl:.2f}")
print(f"2. Gerçekleşmemiş (Açık Pozisyon) Kâr/Zarar: ${toplam_gerçekleşmemiş_pl:.2f}")
print(f"3. Açık Pozisyonların TOPLAM PİYASA DEĞERİ:  ${open_df['Piyasa_Değeri'].sum():.2f}")

# --- B) KAPALI İŞLEMLERİN ANALİZİ ---
print("\n--- 2. KAPALI İŞLEMLER ANALİZİ (Gerçekleşmiş Kâr/Zarar) ---")
display(closed_pl_df)
# Yorum: AAPL ve STX işlemlerinizden elde edilen net kâr/zarar gösterilir.

# --- C) AÇIK POZİSYONLARIN ANALİZİ ---
print("\n--- 3. AÇIK POZİSYONLAR ANALİZİ (Canlı) ---")
display(open_df[['Sembol', 'Net Adet', 'Ort_Alış_Fiyat', 'Canlı_Fiyat', 'Gerçekleşmemiş_P&L', 'Piyasa_Değeri']])

# --- D) GÖRSELLEŞTİRMELER ---

# D.1) Kümülatif P&L Gelişimi
df_cum_pl = df.sort_values('Tarih').copy()
df_cum_pl['Kümülatif P&L'] = df_cum_pl['Nakış Akışı'].cumsum()

fig_cum_pl = px.line(
    df_cum_pl, 
    x='Tarih', 
    y='Kümülatif P&L', 
    title='Kümülatif Nakit Akışı/P&L Gelişimi (İşlem Dönüşü Bazında)',
    markers=True
)
fig_cum_pl.show()

# D.2) Gerçekleşmiş ve Gerçekleşmemiş P&L Karşılaştırması (Yeni Grafik)
pl_comparison = pd.DataFrame({
    'Tip': ['Gerçekleşmiş P&L', 'Gerçekleşmemiş P&L'],
    'Tutar': [toplam_gerçekleşmiş_pl, toplam_gerçekleşmemiş_pl]
})

fig_bar_pl = px.bar(
    pl_comparison,
    x='Tip',
    y='Tutar',
    color='Tutar',
    color_continuous_scale=['red', 'green'],
    title='Toplam Gerçekleşmiş vs. Gerçekleşmemiş Kâr/Zarar'
)
fig_bar_pl.update_layout(showlegend=False)
fig_bar_pl.show()

# D.3) Açık Pozisyonların P&L Dağılımı
fig_unrealized = px.pie(
    open_df, 
    values='Gerçekleşmemiş_P&L', 
    names='Sembol', 
    title='Açık Pozisyonların Gerçekleşmemiş P&L Dağılımı (Canlı Fiyatlarla)'
)
fig_unrealized.show()

Canlı fiyatı çekilecek açık pozisyonlar: ['AAPL', 'BRK.B', 'META', 'NVDA', 'VOO']


$BRK.B: possibly delisted; no price data found  (period=1d) (Yahoo error = "No data found, symbol may be delisted")


UYARI: BRK.B için güncel fiyat bulunamadı. Fiyat 0.0 olarak ayarlandı.

              NASDAQ PORTFÖY VE İŞLEM ANALİZ RAPORU

--- 1. GENEL P&L ÖZETİ (Tüm Zamanlar) ---
1. Gerçekleşmiş (Kapanmış) Toplam Kâr/Zarar: $8.90
2. Gerçekleşmemiş (Açık Pozisyon) Kâr/Zarar: $-115.33
3. Açık Pozisyonların TOPLAM PİYASA DEĞERİ:  $873.99

--- 2. KAPALI İŞLEMLER ANALİZİ (Gerçekleşmiş Kâr/Zarar) ---


,Sembol,Gerçekleşmiş P&L ($)
0,STX,8.9



--- 3. AÇIK POZİSYONLAR ANALİZİ (Canlı) ---


İşlem,Sembol,Net Adet,Ort_Alış_Fiyat,Canlı_Fiyat,Gerçekleşmemiş_P&L,Piyasa_Değeri
0,AAPL,0.026,247.39,263.459991,0.417820,6.849960
1,BRK.B,0.254,492.89,0.000000,-125.194060,0.000000
2,META,0.500,622.57,738.710022,58.070011,369.355011
3,NVDA,1.002,179.57,185.681396,6.123619,186.052759
5,VOO,0.500,732.95,623.455017,-54.747491,311.727509
